In [ ]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import os
import time
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [ ]:
# ==========================================================
# FILE PATHS
# ==========================================================

# Input Files
MASTER_FILE = "/content/drive/MyDrive/Exoplanet_Dataset/08_Master_Pipeline/Master_Pipeline.csv"

TRANSIT_FILE = "/content/drive/MyDrive/Exoplanet_Dataset/09_Transit_Features/Transit_Features.csv"

# Light Curves
LIGHTCURVE_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/05_Preprocessed_LightCurves"

# Output Folder
OUTPUT_FOLDER = "/content/drive/MyDrive/Exoplanet_Dataset/10_Physics_Features"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Output Files
PHYSICS_FILE = os.path.join(
    OUTPUT_FOLDER,
    "Physics_Features.csv"
)

LOG_FILE = os.path.join(
    OUTPUT_FOLDER,
    "Physics_Log.csv"
)

print("=" * 60)
print("PATHS LOADED")
print("=" * 60)

print("Master     :", MASTER_FILE)
print("Transit    :", TRANSIT_FILE)
print("LightCurve :", LIGHTCURVE_FOLDER)
print("Output     :", OUTPUT_FOLDER)

PATHS LOADED
Master     : /content/drive/MyDrive/Exoplanet_Dataset/08_Master_Pipeline/Master_Pipeline.csv
Transit    : /content/drive/MyDrive/Exoplanet_Dataset/09_Transit_Features/Transit_Features.csv
LightCurve : /content/drive/MyDrive/Exoplanet_Dataset/05_Preprocessed_LightCurves
Output     : /content/drive/MyDrive/Exoplanet_Dataset/10_Physics_Features


In [ ]:
# ==========================================================
# CREATE OUTPUT FILES
# ==========================================================

# ----------------------------------------------------------
# Physics Feature File
# ----------------------------------------------------------

if not os.path.exists(PHYSICS_FILE):

    pd.DataFrame(columns=[

        "TIC_ID",

        "ingress_fraction",

        "flat_bottom_fraction",

        "transit_snr",

        "odd_even_ratio",

        "secondary_depth",

        "secondary_snr",

        "duty_cycle"

    ]).to_csv(

        PHYSICS_FILE,

        index=False

    )

# ----------------------------------------------------------
# Log File
# ----------------------------------------------------------

if not os.path.exists(LOG_FILE):

    pd.DataFrame(columns=[

        "TIC_ID",

        "Status",

        "Runtime_Seconds"

    ]).to_csv(

        LOG_FILE,

        index=False

    )

print("=" * 60)
print("OUTPUT FILES READY")
print("=" * 60)

print("Physics File :", PHYSICS_FILE)
print("Log File     :", LOG_FILE)

OUTPUT FILES READY
Physics File : /content/drive/MyDrive/Exoplanet_Dataset/10_Physics_Features/Physics_Features.csv
Log File     : /content/drive/MyDrive/Exoplanet_Dataset/10_Physics_Features/Physics_Log.csv


In [ ]:
# ==========================================================
# LOAD MASTER DATASET
# ==========================================================

master = pd.read_csv(MASTER_FILE)

transit = pd.read_csv(TRANSIT_FILE)

# ----------------------------------------------------------
# Merge
# ----------------------------------------------------------

dataset = master.merge(

    transit,

    on="TIC_ID",

    how="inner"

)

# ----------------------------------------------------------
# Sort
# ----------------------------------------------------------

dataset = dataset.sort_values(

    "TIC_ID"

).reset_index(

    drop=True

)

print("=" * 60)
print("MASTER DATASET LOADED")
print("=" * 60)

print()

print("Rows    :", len(dataset))
print("Columns :", len(dataset.columns))

print()

display(dataset.head())

print()

print("Missing Values")

print(dataset.isna().sum())

MASTER DATASET LOADED

Rows    : 1172
Columns : 17



,TIC_ID,Label,Number_of_Sectors,Best_Sector,period_days,duration_days,epoch_days,bls_depth,bls_power,peak_sigma,depth_ppm,T_tot_hours,T_in_hours,duration_phase,ingress_phase,red_chi2,rms_resid
0,737546,EB,1,5,3.079013,0.183,1441.146152,0.176721,6.905140e+06,9.223567,187384.370864,6.939996,2.196000,0.093915,0.029717,1520.997363,0.057988
1,927579,EB,3,8,7.313719,0.138,1522.313799,0.007182,9.297943e+01,4.108305,7566.254198,3.571631,0.502543,0.020348,0.002863,0.965778,0.007335
2,1528696,Planet,3,98,0.881897,0.030,3994.614357,0.007043,7.459962e+02,43.508619,10147.148822,0.790838,0.304673,0.037364,0.014395,0.996197,0.016563
3,1950736,False Positive,2,73,0.777963,0.210,3285.981604,0.000118,2.110125e+01,7.311826,108.921018,5.841590,2.520000,0.312868,0.134968,0.982580,0.001067
4,2234723,EB,9,43,0.616019,0.249,2474.387552,0.003113,2.089008e+05,25.419678,3023.015425,6.605750,2.902085,0.446803,0.196293,18.013362,0.001676



Missing Values
TIC_ID               0
Label                0
Number_of_Sectors    0
Best_Sector          0
period_days          0
duration_days        0
epoch_days           0
bls_depth            0
bls_power            0
peak_sigma           0
depth_ppm            0
T_tot_hours          0
T_in_hours           0
duration_phase       0
ingress_phase        0
red_chi2             0
rms_resid            0
dtype: int64


In [ ]:
# ==========================================================
# FORMULA VALIDATION
# ==========================================================

planet = dataset[
    dataset["Label"] == "Planet"
].iloc[0]

print("=" * 60)
print("FORMULA VALIDATION")
print("=" * 60)

print()

print("TIC_ID :", int(planet["TIC_ID"]))

print()

print(

    "Ingress Fraction :",

    ingress_fraction(

        planet["T_in_hours"],

        planet["T_tot_hours"]

    )

)

print(

    "Flat Bottom Fraction :",

    flat_bottom_fraction(

        planet["T_in_hours"],

        planet["T_tot_hours"]

    )

)

print(

    "Duty Cycle :",

    duty_cycle(

        planet["T_tot_hours"],

        planet["period_days"]

    )

)

FORMULA VALIDATION

TIC_ID : 1528696

Ingress Fraction : 0.3852534783188438
Flat Bottom Fraction : 0.22949304336231238
Duty Cycle : 0.037364450290535964


In [ ]:
# ==========================================================
# CONFIGURATION
# ==========================================================

MIN_POINTS = 30

print("=" * 60)
print("CONFIGURATION LOADED")
print("=" * 60)

print(f"Minimum Transit Points : {MIN_POINTS}")


# ==========================================================
# PHASE FOLDING
# ==========================================================

def phase_fold(
    time,
    period,
    epoch
):
    """
    Fold observation times into orbital phase.

    Returns
    -------
    Phase in the interval [-0.5, 0.5]
    """

    phase = (

        (

            time

            - epoch

            + 0.5 * period

        )

        % period

    ) / period

    return phase - 0.5

CONFIGURATION LOADED
Minimum Transit Points : 30


In [ ]:
# ==========================================================
# LOAD LIGHT CURVE
# ==========================================================

def load_lightcurve(row):
    """
    Load the cleaned light curve corresponding
    to the best TESS sector.
    """

    tic = int(row["TIC_ID"])

    sector = row["Best_Sector"]

    file = os.path.join(

        LIGHTCURVE_FOLDER,

        f"TIC_{tic}.csv"

    )

    lightcurve = pd.read_csv(file)

    lightcurve = lightcurve[

        lightcurve["Sector"] == sector

    ].copy()

    lightcurve = lightcurve[

        lightcurve["Quality"] == 0

    ].copy()

    return lightcurve

In [ ]:
# ==========================================================
# PREPARE FOLDED LIGHT CURVE
# ==========================================================

def prepare_folded_lightcurve(
    row,
    lightcurve
):
    """
    Prepare a clean phase-folded light curve.
    """

    phase = phase_fold(

        lightcurve["Time"].values,

        row["period_days"],

        row["epoch_days"]

    )

    flux = lightcurve["Normalized_Flux"].values

    flux_error = lightcurve["Normalized_Flux_Error"].values

    good = np.isfinite(flux)

    good &= np.isfinite(flux_error)

    good &= flux_error > 0

    good &= np.abs(flux) < 5

    phase = phase[good]

    flux = flux[good]

    flux_error = flux_error[good]

    order = np.argsort(phase)

    phase = phase[order]

    flux = flux[order]

    flux_error = flux_error[order]

    return phase, flux, flux_error

In [ ]:
# ==========================================================
# LIGHT CURVE VALIDATION
# ==========================================================

planet = dataset[
    dataset["Label"] == "Planet"
].iloc[0]

lightcurve = load_lightcurve(planet)

phase, flux, flux_error = prepare_folded_lightcurve(

    planet,

    lightcurve

)

print("=" * 60)
print("LIGHT CURVE VALIDATION")
print("=" * 60)

print()

print("TIC_ID :", int(planet["TIC_ID"]))

print("Observations :", len(phase))

print("Flux Range :", np.min(flux), "to", np.max(flux))

LIGHT CURVE VALIDATION

TIC_ID : 1528696
Observations : 169754
Flux Range : 0.9227837522715202 to 1.085632395673287


In [ ]:
# ==========================================================
# SHAPE FEATURE EXTRACTION
# ==========================================================

def extract_shape_features(
    row,
    phase,
    flux,
    flux_error
):
    """
    Compute physics-based shape features from the
    folded transit.
    """

    duration_phase = row["duration_phase"]

    transit = np.abs(phase) <= duration_phase / 2

    if transit.sum() < MIN_POINTS:

        return {

            "transit_snr": np.nan,

            "secondary_depth": np.nan,

            "secondary_snr": np.nan

        }

    oot = ~transit

    sigma_oot = np.std(flux[oot])

    if sigma_oot <= 0:

        sigma_oot = np.nan

    # ------------------------------------------------------
    # Transit Depth
    # ------------------------------------------------------

    baseline = np.median(

        flux[oot]

    )

    transit_depth = (

        baseline

        - np.median(

            flux[transit]

        )

    )

    transit_snr = (

        transit_depth

        / sigma_oot

    )

    # ------------------------------------------------------
    # Secondary Eclipse
    # ------------------------------------------------------

    secondary = np.abs(

        np.abs(phase)

        - 0.5

    ) <= duration_phase / 2

    if secondary.sum() >= MIN_POINTS:

        secondary_depth = max(

            0,

            baseline

            - np.median(

                flux[secondary]

            )

        )

        secondary_snr = (

            secondary_depth

            / sigma_oot

        )

    else:

        secondary_depth = np.nan

        secondary_snr = np.nan

    return {

        "transit_snr": transit_snr,

        "secondary_depth": secondary_depth * 1e6,

        "secondary_snr": secondary_snr

    }

In [ ]:
# ==========================================================
# SHAPE FEATURE VALIDATION
# ==========================================================

planet = dataset[
    dataset["Label"] == "Planet"
].iloc[0]

lightcurve = load_lightcurve(planet)

phase, flux, flux_error = prepare_folded_lightcurve(

    planet,

    lightcurve

)

features = extract_shape_features(

    planet,

    phase,

    flux,

    flux_error

)

print("=" * 60)
print("SHAPE FEATURE VALIDATION")
print("=" * 60)

for key, value in features.items():

    print(f"{key:20s}: {value}")

SHAPE FEATURE VALIDATION
transit_snr         : 0.3816557885684248
secondary_depth     : 417.09001285639147
secondary_snr       : 0.025352140642280713


In [ ]:
# ==========================================================
# ODD-EVEN DEPTH RATIO
# ==========================================================

def compute_odd_even_ratio(
    row,
    lightcurve
):
    """
    Compute odd-even transit depth ratio by folding
    the light curve at twice the orbital period.
    """

    phase = phase_fold(

        lightcurve["Time"].values,

        row["period_days"] * 2,

        row["epoch_days"]

    )

    flux = lightcurve["Normalized_Flux"].values

    # ------------------------------------------------------
    # Remove Invalid Values
    # ------------------------------------------------------

    good = np.isfinite(flux)

    good &= np.abs(flux) < 5

    phase = phase[good]

    flux = flux[good]

    duration = row["duration_phase"] / 2

    # ------------------------------------------------------
    # Odd Transit
    # ------------------------------------------------------

    odd = np.abs(phase) <= duration / 2

    # ------------------------------------------------------
    # Even Transit
    # ------------------------------------------------------

    even = np.abs(

        np.abs(phase)

        - 0.5

    ) <= duration / 2

    if odd.sum() < MIN_POINTS:

        return np.nan

    if even.sum() < MIN_POINTS:

        return np.nan

    odd_depth = (

        np.median(

            flux[~odd]

        )

        -

        np.median(

            flux[odd]

        )

    )

    even_depth = (

        np.median(

            flux[~even]

        )

        -

        np.median(

            flux[even]

        )

    )

    if even_depth <= 0:

        return np.nan

    return odd_depth / even_depth

In [ ]:
# ==========================================================
# ODD-EVEN VALIDATION
# ==========================================================

planet = dataset[
    dataset["Label"] == "Planet"
].iloc[0]

lightcurve = load_lightcurve(planet)

ratio = compute_odd_even_ratio(

    planet,

    lightcurve

)

print("=" * 60)
print("ODD-EVEN VALIDATION")
print("=" * 60)

print()

print("TIC_ID :", int(planet["TIC_ID"]))

print("Odd-Even Ratio :", ratio)

ODD-EVEN VALIDATION

TIC_ID : 1528696
Odd-Even Ratio : 1.010676777832686


In [ ]:
# ==========================================================
# PROCESS ONE STAR
# ==========================================================

def process_star(row):
    """
    Generate all physics features for one target.
    """

    # ------------------------------------------------------
    # Formula Features
    # ------------------------------------------------------

    ingress = ingress_fraction(

        row["T_in_hours"],

        row["T_tot_hours"]

    )

    flat = flat_bottom_fraction(

        row["T_in_hours"],

        row["T_tot_hours"]

    )

    duty = duty_cycle(

        row["T_tot_hours"],

        row["period_days"]

    )

    # ------------------------------------------------------
    # Load Light Curve
    # ------------------------------------------------------

    lightcurve = load_lightcurve(row)

    phase, flux, flux_error = prepare_folded_lightcurve(

        row,

        lightcurve

    )

    # ------------------------------------------------------
    # Shape Features
    # ------------------------------------------------------

    shape = extract_shape_features(

        row,

        phase,

        flux,

        flux_error

    )

    # ------------------------------------------------------
    # Odd-Even Ratio
    # ------------------------------------------------------

    try:

        odd_even = compute_odd_even_ratio(

            row,

            lightcurve

        )

    except Exception:

        odd_even = np.nan

    # ------------------------------------------------------
    # Return Features
    # ------------------------------------------------------

    return {

        "TIC_ID": int(row["TIC_ID"]),

        "ingress_fraction": ingress,

        "flat_bottom_fraction": flat,

        "transit_snr": shape["transit_snr"],

        "odd_even_ratio": odd_even,

        "secondary_depth": shape["secondary_depth"],

        "secondary_snr": shape["secondary_snr"],

        "duty_cycle": duty

    }

In [ ]:
# ==========================================================
# FINAL PHYSICS FEATURE VALIDATION
# ==========================================================

classes = [

    "Planet",

    "EB",

    "False Positive"

]

print("=" * 80)
print("FINAL PHYSICS FEATURE VALIDATION")
print("=" * 80)

for label in classes:

    print()

    print("-" * 80)
    print(label.upper())
    print("-" * 80)

    row = dataset[
        dataset["Label"] == label
    ].iloc[0]

    result = process_star(row)

    for key, value in result.items():

        print(f"{key:22s}: {value}")

FINAL PHYSICS FEATURE VALIDATION

--------------------------------------------------------------------------------
PLANET
--------------------------------------------------------------------------------
TIC_ID                : 1528696
ingress_fraction      : 0.3852534783188438
flat_bottom_fraction  : 0.22949304336231238
transit_snr           : 0.3816557885684248
odd_even_ratio        : 1.010676777832686
secondary_depth       : 417.09001285639147
secondary_snr         : 0.025352140642280713
duty_cycle            : 0.037364450290535964

--------------------------------------------------------------------------------
EB
--------------------------------------------------------------------------------
TIC_ID                : 737546
ingress_fraction      : 0.316426706344628
flat_bottom_fraction  : 0.36714658731074407
transit_snr           : 4.386221443452875
odd_even_ratio        : 1.0968831516128774
secondary_depth       : 30812.446591562126
secondary_snr         : 0.9387849944777311
duty_c

In [ ]:
# ==========================================================
# PHYSICS FEATURE GENERATION
# ==========================================================

start = time.perf_counter()

successful = 0
failed = 0

already_processed = set(
    pd.read_csv(PHYSICS_FILE)["TIC_ID"].astype(int)
)

remaining = dataset[
    ~dataset["TIC_ID"].isin(already_processed)
]

print("=" * 60)
print("PHYSICS FEATURE GENERATION STARTED")
print("=" * 60)

print("Already Processed :", len(already_processed))
print("Remaining         :", len(remaining))
print()

for _, row in tqdm(

    remaining.iterrows(),

    total=len(remaining)

):

    tic = int(row["TIC_ID"])

    star_start = time.perf_counter()

    try:

        result = process_star(row)

        pd.DataFrame([result]).to_csv(

            PHYSICS_FILE,

            mode="a",

            header=False,

            index=False

        )

        pd.DataFrame([{

            "TIC_ID": tic,

            "Status": "Success",

            "Runtime_Seconds":

                round(

                    time.perf_counter()

                    - star_start,

                    3

                )

        }]).to_csv(

            LOG_FILE,

            mode="a",

            header=False,

            index=False

        )

        successful += 1

    except Exception as e:

        failed += 1

        print(f"Failed TIC {tic}")

        print(e)

        pd.DataFrame([{

            "TIC_ID": tic,

            "Status": str(e),

            "Runtime_Seconds":

                round(

                    time.perf_counter()

                    - star_start,

                    3

                )

        }]).to_csv(

            LOG_FILE,

            mode="a",

            header=False,

            index=False

        )

print()

print("=" * 60)
print("PHYSICS FEATURE GENERATION COMPLETE")
print("=" * 60)

print("Successful :", successful)
print("Failed     :", failed)

print(

    "Runtime (minutes):",

    round(

        (time.perf_counter() - start)/60,

        2

    )

)

PHYSICS FEATURE GENERATION STARTED
Already Processed : 0
Remaining         : 1172



  0%|          | 0/1172 [00:00<?, ?it/s]


PHYSICS FEATURE GENERATION COMPLETE
Successful : 1172
Failed     : 0
Runtime (minutes): 20.56


In [ ]:
# ==========================================================
# PHYSICS FEATURE SUMMARY
# ==========================================================

physics = pd.read_csv(PHYSICS_FILE)

print("=" * 60)
print("PHYSICS FEATURE SUMMARY")
print("=" * 60)

print()

print("Rows    :", len(physics))
print("Columns :", len(physics.columns))

print()

display(physics.head())

print()

print("Missing Values")

print(physics.isna().sum())

print()

print(physics.describe())

print()

print(

    "Duplicate TIC IDs :",

    physics["TIC_ID"].duplicated().sum()

)

PHYSICS FEATURE SUMMARY

Rows    : 1172
Columns : 8



,TIC_ID,ingress_fraction,flat_bottom_fraction,transit_snr,odd_even_ratio,secondary_depth,secondary_snr,duty_cycle
0,737546,0.316427,0.367147,4.386221,1.096883,30812.446592,0.938785,0.093915
1,927579,0.140704,0.718592,0.868916,NaN,403.933092,0.056066,0.020348
2,1528696,0.385253,0.229493,0.381656,1.010677,417.090013,0.025352,0.037364
3,1950736,0.431389,0.137221,0.107664,5.257057,0.000000,0.000000,0.312868
4,2234723,0.439327,0.121346,2.430875,1.240329,0.000000,0.000000,0.446803



Missing Values
TIC_ID                    0
ingress_fraction          0
flat_bottom_fraction      0
transit_snr              12
odd_even_ratio          137
secondary_depth          21
secondary_snr            21
duty_cycle                0
dtype: int64

             TIC_ID  ingress_fraction  flat_bottom_fraction  transit_snr  \
count  1.172000e+03       1172.000000           1172.000000  1160.000000   
mean   2.332153e+08          0.264782              0.473218     2.918503   
std    1.591498e+08          0.122061              0.235403     3.528645   
min    7.375460e+05          0.025000              0.000000     0.019568   
25%    1.229387e+08          0.161068              0.269732     0.570595   
50%    2.302209e+08          0.289593              0.420814     1.495258   
75%    3.408830e+08          0.365134              0.677864     4.233704   
max    1.965257e+09          0.941250              0.950000    35.423498   

       odd_even_ratio  secondary_depth  secondary_snr   duty_